In [45]:
import pandas as pd

df = pd.read_csv("../data/features.csv")

print(df.shape)
df.head()

(1628, 11)


,hold_mean,hold_std,hold_min,hold_max,flight_mean,flight_std,flight_min,flight_max,digraph_mean,digraph_std,PARTICIPANT_ID
0,105.40,19.693440,56.0,136.0,239.20,358.878458,7.0,1632.0,344.60,360.343462,10004
1,110.30,28.545624,64.0,184.0,174.95,192.746486,8.0,577.0,285.25,197.036966,10004
2,105.10,23.621801,56.0,136.0,161.30,182.732505,8.0,568.0,266.40,175.458317,10004
3,118.10,50.166984,64.0,306.0,204.20,221.509083,4.0,776.0,322.30,206.897661,10004
4,103.85,14.747970,80.0,136.0,158.10,175.234249,8.0,721.0,261.95,172.637885,10004


In [46]:
from sklearn.preprocessing import LabelEncoder

X = df.drop("PARTICIPANT_ID", axis=1)
y = df["PARTICIPANT_ID"]

le = LabelEncoder()
y = le.fit_transform(y)

print("Total classes:", len(set(y)))

Total classes: 50


In [47]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [48]:
import pandas as pd

# Count samples per user
counts = pd.Series(y).value_counts()

# Keep only users with >= 2 samples
valid_classes = counts[counts >= 2].index

mask = [label in valid_classes for label in y]

X = X[mask]
y = y[mask]

print("After filtering:")
print("Samples:", len(y))
print("Classes:", len(set(y)))

After filtering:
Samples: 1628
Classes: 50


In [49]:
# Re-encode labels after filtering
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [51]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_split=5,
        random_state=42
    ),
    
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss',
        random_state=42
    )
}

results = {}

for name, model in models.items():
    print(f"\n Training {name}...")
    
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, pred)
    results[name] = acc
    
    print(f"✅ {name} Accuracy: {acc:.4f}")


 Training Random Forest...
✅ Random Forest Accuracy: 0.3650

 Training XGBoost...
✅ XGBoost Accuracy: 0.4294


In [52]:
from sklearn.metrics import top_k_accuracy_score

xgb_model = models["XGBoost"]

pred_proba = xgb_model.predict_proba(X_test)

top3_acc = top_k_accuracy_score(y_test, pred_proba, k=3)

print(" Top-3 Accuracy:", top3_acc)

 Top-3 Accuracy: 0.6871165644171779


In [53]:
import pickle

# Save best model (XGBoost)
with open("../models/xgb_model.pkl", "wb") as f:
    pickle.dump(models["XGBoost"], f)

print("✅ XGBoost model saved successfully")

✅ XGBoost model saved successfully


In [54]:
# Save scaler
with open("../models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Save label encoder
with open("../models/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("✅ Scaler + LabelEncoder saved")

✅ Scaler + LabelEncoder saved
